# Steam Game Popularity Predictor – Stage 1 EDA Report
### University Course: Programming for AI
This notebook performs exploratory data analysis directly from the validated SQLite database (data/steam.db).

All 9 required statistical tables and visualizations are generated below:
1. Summary Statistics & High Skewness
2. Missing Value Analysis & Outlier Detection (IQR)
3. Genre Performance by Popularity Tier
4. Price Band Distribution
5. Free-to-Play vs. Paid Dynamics
6. Platform Support Synergy
7. Yearly Trends (2000–2019)
8. Feature Correlation Matrix & Leakage Analysis
9. Popularity Tier Target Distribution (Class Imbalance)


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 12

# Connect to database
conn = sqlite3.connect('../data/steam.db')
df = pd.read_sql_query('SELECT * FROM games', conn)
print(f'Successfully loaded {len(df):,} games with {len(df.columns)} columns.')


## 1. Summary Statistics
Examines distribution, variance, spread, and extreme positive skewness in commercial metrics.


In [ ]:
cols = ['price', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners_mid']
summary = df[cols].describe().T[['mean', 'std', 'min', '50%', 'max']]
summary['skewness'] = df[cols].skew()
summary.rename(columns={'50%': 'median'}, inplace=True)
summary


## 2. Missing Values & Outlier Analysis (IQR Method)
Checking completeness across all columns and calculating outliers using 1.5 * IQR.


In [ ]:
outlier_records = []
for c in cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((df[c] < lower) | (df[c] > upper)).sum())
    pct = round((count / len(df)) * 100, 2)
    outlier_records.append({'metric': c, 'q1': round(q1, 2), 'q3': round(q3, 2), 'iqr': round(iqr, 2), 'outliers': count, 'pct_outliers': pct})

outliers_df = pd.DataFrame(outlier_records)
outliers_df


## 3. Popularity Tier Distribution (Target Variable)
Examines class proportions across Low (<= 20k), Medium (20k-200k), and High (> 200k).


In [ ]:
tier_counts = df['popularity_tier'].value_counts()[['Low', 'Medium', 'High']]
tier_pct = (tier_counts / len(df)) * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Donut chart
colors = ['#3b82f6', '#10b981', '#f59e0b']
ax[0].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', startangle=140, colors=colors, wedgeprops=dict(width=0.4, edgecolor='w'))
ax[0].set_title('Target Class Distribution')

# Bar plot
sns.barplot(x=tier_counts.index, y=tier_counts.values, palette=colors, ax=ax[1], hue=tier_counts.index, legend=False)
for i, v in enumerate(tier_counts.values):
    ax[1].text(i, v + 20, f'{v} ({tier_pct.iloc[i]:.1f}%)', ha='center', fontweight='bold')
ax[1].set_ylabel('Number of Games')
ax[1].set_title('Game Count by Tier')
plt.tight_layout()
plt.show()


## 4. Genre Performance by Popularity Tier
Analyzes which game genres yield the highest concentration of High-tier commercial successes.


In [ ]:
# Query genres many-to-many
genre_query = '''
SELECT g.name AS genre, gm.popularity_tier, COUNT(*) as count
FROM game_genres gg
JOIN genres g ON gg.genre_id = g.id
JOIN games gm ON gg.game_id = gm.appid
GROUP BY g.name, gm.popularity_tier
'''
genre_df = pd.read_sql_query(genre_query, conn)
genre_pivot = genre_df.pivot(index='genre', columns='popularity_tier', values='count').fillna(0)[['Low', 'Medium', 'High']]
genre_pivot['Total'] = genre_pivot.sum(axis=1)
genre_pivot['High_Pct'] = (genre_pivot['High'] / genre_pivot['Total']) * 100
top_genres = genre_pivot.sort_values(by='Total', ascending=False).head(10)

plt.figure(figsize=(12, 6))
top_genres[['Low', 'Medium', 'High']].plot(kind='bar', stacked=True, color=colors, figsize=(12, 6))
plt.title('Top 10 Genres by Tier Volume')
plt.xlabel('Genre')
plt.ylabel('Number of Games')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Price Band Distribution
Assessing commercial success rate across different pricing segments.


In [ ]:
bins = [-1, 0, 4.99, 9.99, 19.99, 29.99, float('inf')]
labels = ['Free ()', '.01 - .99', '.00 - .99', '.00 - .99', '.00 - .99', '.00+']
df['price_band'] = pd.cut(df['price'], bins=bins, labels=labels)

price_tier = pd.crosstab(df['price_band'], df['popularity_tier'], normalize='index')[['Low', 'Medium', 'High']] * 100

plt.figure(figsize=(12, 5))
price_tier.plot(kind='bar', stacked=True, color=colors, figsize=(12, 5))
plt.title('Popularity Tier Composition by Price Band')
plt.xlabel('Price Band')
plt.ylabel('Percentage within Band (%)')
plt.xticks(rotation=30)
plt.legend(title='Tier')
plt.tight_layout()
plt.show()


## 6. Free-to-Play vs. Paid Comparison
Comparing conversion rates, ratings, and playtime between Free and Paid titles.


In [ ]:
f2p_stats = df.groupby('is_free').agg(
    total=('appid', 'count'),
    high_count=('popularity_tier', lambda x: (x == 'High').sum()),
    avg_price=('price', 'mean'),
    avg_review_score=('review_score', 'mean'),
    median_playtime=('median_playtime', 'median')
).reset_index()
f2p_stats['is_free'] = f2p_stats['is_free'].map({0: 'Paid', 1: 'Free-to-Play'})
f2p_stats['high_conversion_pct'] = (f2p_stats['high_count'] / f2p_stats['total']) * 100
f2p_stats


## 7. Platform Support Synergy
Impact of multi-platform OS availability (Windows, macOS, Linux) on commercial tier distribution.


In [ ]:
df['os_combo'] = df.apply(
    lambda r: '+'.join([os for os, has in [('Win', r['is_windows']), ('Mac', r['is_mac']), ('Lin', r['is_linux'])] if has]) or 'None',
    axis=1
)
platform_crosstab = pd.crosstab(df['os_combo'], df['popularity_tier'], margins=True)
platform_crosstab


## 8. Yearly Trends (2000–2019)
Historical release distribution and median price evolution.


In [ ]:
yearly = df[df['release_year'] >= 2000].groupby('release_year').agg(
    total_releases=('appid', 'count'),
    median_price=('price', 'median')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

sns.barplot(data=yearly, x='release_year', y='total_releases', ax=ax1, color='#93c5fd', alpha=0.7)
sns.lineplot(data=yearly, x=range(len(yearly)), y='median_price', ax=ax2, color='#ef4444', marker='o', linewidth=2.5)

ax1.set_title('Yearly Release Volume & Median Price (2000–2019)')
ax1.set_xlabel('Release Year')
ax1.set_ylabel('Annual Release Count', color='#1e3a8a')
ax2.set_ylabel('Median Price ($)', color='#ef4444')
ax1.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 9. Feature Correlation Matrix & Leakage Analysis
Checking linear relationships between features. Post-launch metrics are separated to prevent data leakage in ML stages.


In [ ]:
corr_cols = ['price', 'achievements', 'is_free', 'is_windows', 'is_mac', 'is_linux', 'has_singleplayer', 'has_multiplayer', 'positive_ratings', 'negative_ratings', 'review_score', 'average_playtime', 'owners_mid']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix (Pre-launch vs. Post-launch Outcomes)')
plt.tight_layout()
plt.show()
